# Reproducible Multi-Seed Experiment & Sensitivity Analysis
## Shift Workload-Balancing Simulator for Hospital Patient Transfers

This notebook executes the reproducible multi-seed experiment across 10 random seeds (`seed=1` to `seed=10`), computing mean metrics, standard deviation, imbalance reduction %, and comprehensive error analysis.

In [1]:
import os
import sys
import pandas as pd
import numpy as np

# Ensure project root path
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.data_cleaning import run_cleaning_pipeline
from src.simulator import run_scenario_simulation, run_all_operating_scenarios, run_sensitivity_analysis, run_reproducible_experiment

# 1. Load Baseline Data
df_units, df_nurses, df_patients = run_cleaning_pipeline(raw_dir="../data/raw", clean_dir="../data/clean")
print(f"Baseline dataset loaded: {len(df_patients)} patients, {len(df_nurses)} nurses, {len(df_units)} units.")

## 1. 8 Operating Scenarios Experiment Table

We evaluate 8 operating scenarios covering Baseline, High Acuity (+25%), Staff Shortages (-20%), Combined Stress, Transfer Surges (+35%), Specialist Shortages (-50%), Bed Capacity Constraints (-25%), and Multi-Unit Stress.

In [2]:
results, summary_df = run_all_operating_scenarios(df_units, df_nurses, df_patients)
display(summary_df[[
    "Scenario_Name", "Pre_Staffing_Gap", "Post_Staffing_Gap",
    "Pre_Workload_Imbalance_Std", "Post_Workload_Imbalance_Std",
    "Imbalance_Reduction_Pct", "Safe_Reassignment_Capacity", "Unsafe_Attempts_Prevented"
]])

## 2. Reproducible Multi-Seed Experiment (10 Random Seeds)

To ensure experimental reproducibility, we execute the simulation across 10 distinct synthetic dataset seeds and calculate mean and standard deviation.

In [3]:
seed_raw_df, exp_metrics_summary = run_reproducible_experiment(num_seeds=10)
print("=== Multi-Seed Experiment (10 Seeds Summary) ===")
display(exp_metrics_summary)

## 3. Comprehensive Error Analysis

### Primary Failure Conditions Analyzed:
1. **Donor Staffing Minimum Breach**: Blocked transfers where donor unit active staff would drop below `Minimum_Staff`.
2. **Physical Bed Capacity Bottleneck**: Blocked transfers where receiving unit bed occupancy is 100% (`Occupied_Beds == Bed_Capacity`).
3. **Skill Mismatch**: Blocked reassignments where donor nurses lack required specialty certification (e.g. Med-Surg nurse requested for ICU).
4. **Severe Multi-Unit Exhaustion**: When all units hit minimum staffing, safe reassignment capacity drops to 0, requiring administrative escalation.